In [1]:
import numpy as np
import tensorflow as tf

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = make_moons(
    n_samples=1200,
    noise=0.25,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [3]:
def build_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    return model

optimizers = {
    "SGD" : tf.keras.optimizers.SGD(learning_rate=0.01),

    "momentum" : tf.keras.optimizers.SGD(
        learning_rate=0.01,
        momentum=0.9
    ),

    "RMSprop" : tf.keras.optimizers.RMSprop(learning_rate=0.01),

    "Adam" : tf.keras.optimizers.Adam(learning_rate=0.01)
}

In [5]:
results = []

for name, optimizer in optimizers.items():
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    model = build_model()

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.20,
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

    probabilities = model.predict(X_test, verbose=0).ravel()

    predictions = (probabilities >= 0.5).astype(int)

    test_accuracy_manual = accuracy_score(y_test, predictions)

    results.append({
        "Optimizer" : name,
        "Final Training Loss" : round(
            history.history['loss'][-1], 4
        ),
        "Final Validation Loss" : round(
            history.history['val_loss'][-1], 4
        ),
        "Test Loss" : round(test_loss, 4),
        "Test Accuracy" : round(test_accuracy_manual, 4)
    })


print("\nOptimizer Comparision")
for result in results:
    print(result)


Optimizer Comparision
{'Optimizer': 'SGD', 'Final Training Loss': 0.2958, 'Final Validation Loss': 0.3701, 'Test Loss': 0.292, 'Test Accuracy': 0.8875}
{'Optimizer': 'momentum', 'Final Training Loss': 0.1536, 'Final Validation Loss': 0.2475, 'Test Loss': 0.1426, 'Test Accuracy': 0.95}
{'Optimizer': 'RMSprop', 'Final Training Loss': 0.1604, 'Final Validation Loss': 0.2716, 'Test Loss': 0.1388, 'Test Accuracy': 0.9458}
{'Optimizer': 'Adam', 'Final Training Loss': 0.1485, 'Final Validation Loss': 0.2497, 'Test Loss': 0.1372, 'Test Accuracy': 0.9583}
